In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler


In [3]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [4]:
# --- CONFIGURATION ---
# Path provided by user
DATA_DIR = r"D:\studies\LPU\sem_2\CSE275_OPTIMIZATION_TECHINIQUES _FOR _ML\Traffic-Congestion-Predictor-Using-ML\METR-LA"
H5_PATH = os.path.join(DATA_DIR, "METR-LA.h5")
ADJ_PATH = os.path.join(DATA_DIR, "adj_METR-LA.pkl")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001
SEQ_LEN = 12  # 1 hour of historical data (5-min intervals)
PRED_LEN = 12 # Predict next 1 hour

def load_data():
    """Load METR-LA traffic speed data and adjacency matrix."""
    # Load speed data
    df = pd.read_hdf(H5_PATH)
    data = df.values.astype(np.float32)
    
    # Load adjacency matrix
    with open(ADJ_PATH, 'rb') as f:
        sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')
    
    return data, adj_mx



def create_sequences(data, seq_len, pred_len):
    #Transform time series into supervised learning format."""
    x, y = [], []
    for i in range(len(data) - seq_len - pred_len):
        x.append(data[i : i + seq_len])
        y.append(data[i + seq_len : i + seq_len + pred_len])
    return np.array(x), np.array(y)

In [5]:
    # --- MODEL ARCHITECTURE (STGCN-like) ---
    class SpatioTemporalBlock(nn.Module):
    # Simplified Spatio-Temporal Block for Traffic Prediction."""
        def __init__(self, in_channels, spatial_channels, out_channels, num_nodes, adj):
            super(SpatioTemporalBlock, self).__init__()
            self.adj = torch.from_numpy(adj).to(DEVICE).float()
            # Temporal Convolution (1D Conv over time)
            self.temporal1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 1), padding=(1, 0))
            
            # Spatial Graph Convolution (Approximated)
            self.spatial = nn.Linear(num_nodes, num_nodes)
            
            # Temporal Convolution 2
            self.temporal2 = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 1), padding=(1, 0))
            
            self.relu = nn.ReLU()
            self.batch_norm = nn.BatchNorm2d(out_channels)

        def forward(self, x):
            # x shape: [Batch, Channels, Time, Nodes]
            x = self.temporal1(x)
            
            # Apply adjacency matrix interaction (Spatial)
            # Reshape for matrix mult: [B, C, T, N] -> [B, C, T, N]
            x = torch.matmul(x, self.adj) 
            
            x = self.temporal2(x)
            return self.batch_norm(self.relu(x))

In [6]:
    class TrafficGNN(nn.Module):
        def __init__(self, num_nodes, adj):
            super(TrafficGNN, self).__init__()
            self.st_block1 = SpatioTemporalBlock(1, 16, 32, num_nodes, adj)
            self.st_block2 = SpatioTemporalBlock(32, 32, 64, num_nodes, adj)
            
            # Final output layer to predict PRED_LEN time steps
            self.output = nn.Sequential(
                nn.Flatten(),
                nn.Linear(64 * SEQ_LEN * num_nodes, num_nodes * PRED_LEN)
            )
            self.num_nodes = num_nodes
            
        def forward(self, x):
            # Input x: [Batch, Seq, Nodes] -> Add channel dim: [Batch, 1, Seq, Nodes]
            x = x.unsqueeze(1)
            x = self.st_block1(x)
            x = self.st_block2(x)
            out = self.output(x)
            return out.view(-1, PRED_LEN, self.num_nodes)

In [7]:
    # --- EVALUATION METRIC ---
    def calculate_rmse(y_true, y_pred):
        return torch.sqrt(nn.MSELoss()(y_true, y_pred))

In [10]:
    # --- MAIN EXECUTION ---
    def train():
        print(f"Loading data from {DATA_DIR}...")
        raw_data, adj_mx = load_data()
        num_nodes = raw_data.shape[1]
        
        # Scale data
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(raw_data)
        
        # Create sequences
        X, Y = create_sequences(scaled_data, SEQ_LEN, PRED_LEN)
            
        # Split: 70% Train, 15% Val, 15% Test
        train_size = int(0.7 * len(X))
        val_size = int(0.15 * len(X))
        
        train_x, train_y = torch.tensor(X[:train_size]), torch.tensor(Y[:train_size])
        val_x, val_y = torch.tensor(X[train_size:train_size+val_size]), torch.tensor(Y[train_size:train_size+val_size])
        test_x, test_y = torch.tensor(X[train_size+val_size:]), torch.tensor(Y[train_size+val_size:])
        
        train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=BATCH_SIZE)


        model = TrafficGNN(num_nodes, adj_mx).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.MSELoss()
    
        print("Starting training focusing on RMSE minimization...")
        best_val_rmse = float('inf')

        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                
                optimizer.zero_grad()
                output = model(batch_x)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
    
                # Validation
                model.eval()
                val_rmse = 0
                with torch.no_grad():
                    for bx, by in val_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        pred = model(bx)
                        val_rmse += calculate_rmse(by, pred).item()
    
                avg_val_rmse = val_rmse / len(val_loader)
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val RMSE: {avg_val_rmse:.4f}")
            
                if avg_val_rmse < best_val_rmse:
                    best_val_rmse = avg_val_rmse
                    torch.save(model.state_dict(), "best_traffic_model.pth")

        print(f"Training Complete. Best Validation RMSE: {best_val_rmse:.4f}")

   
